# 16 — Preparing the fusion detector for training

Notebook 15 produced the model-ready tensor:

```text
7 × 350 × 400
```

This notebook verifies the next boundary:

```text
fused KITTI sample
      ↓
seven-channel batch
      ↓
BEVDetector(input_channels=7)
      ↓
predictions
      ↓
existing detection loss
      ↓
finite gradients
```

It intentionally performs only one optimization step. Full training should begin only after this smoke test succeeds.

## 1. Imports and reproducible setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset

CURRENT_DIRECTORY = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIRECTORY if (CURRENT_DIRECTORY / "src").is_dir() else CURRENT_DIRECTORY.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset.kitti_dataset import KittiDataset
from src.geometry.boxes import get_lidar_boxes
from src.losses.detection_loss import DetectionLoss
from src.models.bev_detector import BEVDetector
from src.preprocessing.fusion import FUSED_BEV_CHANNELS, build_fused_bev
from src.targets.bev_targets import create_bev_targets
from src.training.training_script import collate_detection_batch, seed_everything

SEED = 42
seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [2]:
dataset = KittiDataset(
    root=PROJECT_ROOT / "data" / "KITTI",
    split="training",
    load_images=True,
)

print("Samples:", len(dataset))
print("Images enabled:", dataset.load_images)
print("Labels available:", dataset.has_labels)

Samples: 7481
Images enabled: True
Labels available: True


## 2. Fusion training view

The underlying KITTI dataset still returns raw sensor data. This small view converts each selected sample into the seven-channel BEV and creates the same detection targets used by the LiDAR-only model.

Augmentation is intentionally disabled here. A future training integration must transform points, image correspondence, camera BEV, boxes, and targets consistently.

In [3]:
class FusionDetectionView(Dataset):
    """Create fused BEV tensors and detection targets."""

    def __init__(self, dataset, indices):
        if not dataset.load_images:
            raise ValueError("fusion requires load_images=True")
        if not dataset.has_labels:
            raise ValueError("fusion training requires labels")
        self.dataset = dataset
        self.indices = list(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        sample = self.dataset[self.indices[position]]

        fused_bev = build_fused_bev(
            points=sample["points"],
            image=sample["image"],
            calibration=sample["calib"],
        )

        boxes = get_lidar_boxes(
            sample["labels"],
            sample["calib"],
        )
        targets = create_bev_targets(boxes)
        return fused_bev, targets

In [4]:
fusion_view = FusionDetectionView(
    dataset=dataset,
    indices=[0],
)

inputs, targets = collate_detection_batch(
    [fusion_view[0]]
)

print("Input batch:", inputs.shape, inputs.dtype)
for key, value in targets.items():
    print(f"{key:16s}", tuple(value.shape), value.dtype)

assert inputs.shape == (1, 7, 350, 400)
assert torch.isfinite(inputs).all()
assert all(torch.isfinite(value).all() for value in targets.values())

Input batch: torch.Size([1, 7, 350, 400]) torch.float32
heatmap          (1, 1, 350, 400) torch.float32
offset           (1, 2, 350, 400) torch.float32
size             (1, 2, 350, 400) torch.float32
rotation         (1, 2, 350, 400) torch.float32
regression_mask  (1, 1, 350, 400) torch.float32


### Input channel meaning

In [5]:
for channel_index, channel_name in enumerate(FUSED_BEV_CHANNELS):
    channel = inputs[0, channel_index]
    print(
        f"{channel_index}: {channel_name:20s}",
        f"min={channel.min().item():.3f}",
        f"max={channel.max().item():.3f}",
        f"nonzero={torch.count_nonzero(channel).item()}",
    )

0: lidar_height         min=0.000 max=0.824 nonzero=6324
1: lidar_density        min=0.000 max=1.000 nonzero=6324
2: lidar_intensity      min=0.000 max=0.990 nonzero=5976
3: camera_red           min=0.000 max=1.000 nonzero=2611
4: camera_green         min=0.000 max=1.000 nonzero=2611
5: camera_blue          min=0.000 max=1.000 nonzero=2611
6: camera_visibility    min=0.000 max=1.000 nonzero=2611


## 3. Compare the LiDAR and fusion detector configurations

The architecture is unchanged except for the first convolution. The three-channel constructor remains the default for checkpoint compatibility.

In [6]:
lidar_model = BEVDetector(input_channels=3)
fusion_model = BEVDetector(input_channels=7)

lidar_parameters = sum(parameter.numel() for parameter in lidar_model.parameters())
fusion_parameters = sum(parameter.numel() for parameter in fusion_model.parameters())

print("LiDAR model parameters:", f"{lidar_parameters:,}")
print("Fusion model parameters:", f"{fusion_parameters:,}")
print("Additional parameters:", f"{fusion_parameters - lidar_parameters:,}")
print(
    "Increase:",
    f"{100 * (fusion_parameters - lidar_parameters) / lidar_parameters:.3f}%",
)

assert lidar_model.stem[0].in_channels == 3
assert fusion_model.stem[0].in_channels == 7

LiDAR model parameters: 823,463
Fusion model parameters: 824,615
Additional parameters: 1,152
Increase: 0.140%


Only the first convolution receives additional weights:

```text
LiDAR model:  3 input channels → 32 filters
Fusion model: 7 input channels → 32 filters
```

The encoder, context layer, decoder, prediction heads, targets, loss, and decoder remain unchanged.

## 4. Forward-pass smoke test

In [7]:
fusion_model = fusion_model.to(device)
inputs = inputs.to(device)
targets = {
    key: value.to(device)
    for key, value in targets.items()
}

fusion_model.train()
outputs = fusion_model(inputs)

for key, value in outputs.items():
    print(
        f"{key:10s}",
        tuple(value.shape),
        "finite=",
        torch.isfinite(value).all().item(),
    )

assert outputs["heatmap"].shape == (1, 1, 350, 400)
assert outputs["offset"].shape == (1, 2, 350, 400)
assert outputs["size"].shape == (1, 2, 350, 400)
assert outputs["rotation"].shape == (1, 2, 350, 400)
assert all(torch.isfinite(value).all() for value in outputs.values())

heatmap    (1, 1, 350, 400) finite= True
offset     (1, 2, 350, 400) finite= True
size       (1, 2, 350, 400) finite= True
rotation   (1, 2, 350, 400) finite= True


Random predictions are expected here. The purpose is only to prove that the seven-channel representation passes through every model stage and retains the existing output format.

## 5. Loss and backward-pass smoke test

In [8]:
criterion = DetectionLoss().to(device)
optimizer = torch.optim.AdamW(
    fusion_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

optimizer.zero_grad(set_to_none=True)
losses = criterion(outputs, targets)

for key, value in losses.items():
    print(f"{key:10s}", f"{value.item():.6f}")

assert all(torch.isfinite(value) for value in losses.values())

losses["total"].backward()

gradient_values = [
    parameter.grad.detach().abs().max()
    for parameter in fusion_model.parameters()
    if parameter.grad is not None
]
maximum_gradient = torch.stack(gradient_values).max().item()

print("Parameters with gradients:", len(gradient_values))
print("Maximum absolute gradient:", maximum_gradient)

assert gradient_values
assert np.isfinite(maximum_gradient)

torch.nn.utils.clip_grad_norm_(
    fusion_model.parameters(),
    max_norm=10.0,
)
optimizer.step()

print("One optimizer step completed successfully.")

total      166.941864
heatmap    166.941864
offset     0.000000
size       0.000000
rotation   0.000000
Parameters with gradients: 49
Maximum absolute gradient: 456.78277587890625
One optimizer step completed successfully.


## 6. What this proves—and what it does not

This proves:

- the dataset can produce aligned seven-channel samples;
- batching preserves `(B, 7, 350, 400)`;
- the detector accepts seven channels;
- output and target shapes remain compatible;
- the existing loss is finite;
- gradients flow through the fusion model.

This does **not** prove that RGB improves detection. That requires a controlled training comparison.

## 7. Required production-training integration

The current training command still builds LiDAR-only BEVs. The next source-code change should add:

1. `--input-mode lidar|fusion`;
2. `load_images=True` in fusion mode;
3. a production fusion detection view;
4. `BEVDetector(input_channels=3 or 7)`;
5. checkpoint metadata containing `input_mode` and `input_channels`;
6. synchronized augmentation for all seven channels and targets.

The first experiment should use the exact same seed and split as the LiDAR baseline.

## 8. Safe training progression

In [9]:
recommended_experiments = [
    {
        "stage": "Tiny overfit",
        "samples": 16,
        "epochs": "20–50",
        "validation": 0.0,
        "flip_probability": 0.0,
        "purpose": "Verify that the model can memorize a tiny fused dataset",
    },
    {
        "stage": "Training smoke test",
        "samples": "100–300",
        "epochs": "3–5",
        "validation": 0.2,
        "flip_probability": 0.0,
        "purpose": "Verify loaders, checkpoints, validation, memory, and speed",
    },
    {
        "stage": "Controlled full run",
        "samples": "all",
        "epochs": 30,
        "validation": 0.2,
        "flip_probability": "after synchronized-flip validation",
        "purpose": "Compare fusion against the LiDAR-only baseline",
    },
]

for experiment in recommended_experiments:
    print(experiment)

{'stage': 'Tiny overfit', 'samples': 16, 'epochs': '20–50', 'validation': 0.0, 'flip_probability': 0.0, 'purpose': 'Verify that the model can memorize a tiny fused dataset'}
{'stage': 'Training smoke test', 'samples': '100–300', 'epochs': '3–5', 'validation': 0.2, 'flip_probability': 0.0, 'purpose': 'Verify loaders, checkpoints, validation, memory, and speed'}
{'stage': 'Controlled full run', 'samples': 'all', 'epochs': 30, 'validation': 0.2, 'flip_probability': 'after synchronized-flip validation', 'purpose': 'Compare fusion against the LiDAR-only baseline'}


## Acceptance result

When every cell above succeeds, the model itself is ready for fusion training.

The immediate next implementation is the training-script integration—not a full 30-epoch run. After that integration, first run the 16-sample overfit experiment and visually decode its predictions.